# Stage 1 physical recapture experiment gate

목표는 합성 artefact가 아니라 실제 화면 재촬영을 식별하는 것이다. Stage 1은 가중치 0.2이므로 Stage 3/2 이후에 실행한다.

In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/블랙박스 영상 기반 사고 분석')
PROJECT_ROOT = Path('/content/dacon236753')
EXPERIMENT = 's1_forensics_v1'
MANIFEST = DRIVE_ROOT / 'external_data' / 'stage1' / 'recapture_manifest.csv'
ARTIFACT = DRIVE_ROOT / 'experiments' / EXPERIMENT
for folder in ('config','checkpoints','predictions','reports','submission','logs'):
    (ARTIFACT / folder).mkdir(parents=True, exist_ok=True)
print('Expected:', MANIFEST)
print('Experiment:', ARTIFACT)

## manifest 계약

필수 열: `id,source_clip_id,video_path,label,capture_method,device_group`.

- `source_clip_id` 기준 group split: 동일 원본과 그 재촬영본은 train/validation에 절대 함께 두지 않는다.
- `capture_method`는 original, physical_recapture, synthetic_recapture를 분리 기록한다.
- physical_recapture가 validation 양성 class의 최소 절반을 차지하지 않으면 합성 과적합 위험으로 No-Go다.

In [ ]:
if not MANIFEST.exists():
    print('STOP: 재촬영 manifest가 필요합니다.')
else:
    data = pd.read_csv(MANIFEST)
    required = {'id','source_clip_id','video_path','label','capture_method','device_group'}
    assert not (required - set(data.columns))
    assert set(data.label) <= {'ORIGINAL','RERECORDED'}
    print(data.groupby(['label','capture_method']).size())

## V1 모델과 판단

전체 영상의 3–8 clip temporal RGB encoder와 high-frequency residual branch를 late fusion한다. 물리 재촬영의 display 밝기, 각도, 거리, 반사, moiré, rolling banding을 train augmentation에도 반영한다. source_clip group-holdout Macro-F1가 기준선보다 개선될 때만 후보로 승격한다.

In [ ]:
# FINAL SUBMISSION GATE — 이 셀은 항상 마지막에 둡니다.
import subprocess, sys
BASELINE_LOCAL = ARTIFACT / 'reports/baseline_local_metrics.json'
CANDIDATE_LOCAL = ARTIFACT / 'reports/local_metrics.json'
ZIP_SMOKE_MARKER = ARTIFACT / 'reports/zip_smoke_pass.json'
gate = [sys.executable, str(PROJECT_ROOT / 'tools/go_no_go.py'), '--stage', 'stage1', '--baseline-local', str(BASELINE_LOCAL), '--candidate-local', str(CANDIDATE_LOCAL), '--evaluation-kind', 'external_group_holdout', '--json-out', str(ARTIFACT / 'reports/submission_gate.json')]
if ZIP_SMOKE_MARKER.exists(): gate.append('--smoke-pass')
subprocess.run(gate, check=False)
